# Laravel Docs RAG

RAG over Laravel documentation using:
- **LangChain** `DirectoryLoader` + `RecursiveCharacterTextSplitter` + Chroma
- **Ollama** for embeddings (`nomic-embed-text`) and chat (`llama3.2`)
- Optional **query rewrite** and **rerank**
- **Gradio** chat UI (run the Chat cell below)

Ensure `laravel-docs/` exists in this folder

In [ ]:
import os
from pathlib import Path

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

import re
import gradio as gr
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.documents import Document

ROOT = Path.cwd()
DOCS_PATH = ROOT / "laravel-docs"
DB_NAME = str(ROOT / "chroma_db")
COLLECTION_NAME = "laravel_docs"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 200
embeddings = OllamaEmbeddings(model="nomic-embed-text")

## 1. Ingest

Load markdown from `laravel-docs/`, split with RecursiveCharacterTextSplitter, embed with Ollama, store in Chroma. Re-run to rebuild the index.

In [ ]:
def fetch_documents():
    if not DOCS_PATH.is_dir():
        raise FileNotFoundError(f"Docs path not found: {DOCS_PATH}")
    loader = DirectoryLoader(
        str(DOCS_PATH),
        glob="**/*.md",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
    )
    documents = loader.load()
    print(f"Loaded {len(documents)} documents")
    return documents

def create_chunks(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    chunks = text_splitter.split_documents(documents)
    print(f"Created {len(chunks)} chunks")
    return chunks

def create_embeddings(chunks):
    if os.path.exists(DB_NAME):
        Chroma(
            persist_directory=DB_NAME,
            embedding_function=embeddings,
            collection_name=COLLECTION_NAME,
        ).delete_collection()
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_NAME,
        collection_name=COLLECTION_NAME,
    )
    print(f"Vectorstore created with {vectorstore._collection.count():,} documents in {DB_NAME}")
    return vectorstore

documents = fetch_documents()
chunks = create_chunks(documents)
create_embeddings(chunks)
print("Ingestion complete.")

## 2. Chat

Launch the Gradio chat interface. Ask questions about Laravel; the app uses retrieval, optional query rewrite, and rerank.

In [ ]:
RETRIEVAL_K = 20
FINAL_K = 10
USE_REWRITE = True
USE_RERANK = True
CHAT_MODEL = "llama3.2"

vectorstore = Chroma(
    persist_directory=DB_NAME,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
)
retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})
llm = ChatOllama(temperature=0, model=CHAT_MODEL)

SYSTEM_PROMPT = """You are a helpful assistant that answers questions about Laravel based on the official documentation.
Use only the context below. If the context doesn't contain enough information, say so.
Be accurate, relevant and complete.

Context:
{context}

Answer the user's question based on this context."""

def fetch_context_unranked(question):
    return retriever.invoke(question)

def rewrite_query(question, history):
    history_text = "None"
    if history:
        parts = [f"User: {m[0]}\nAssistant: {m[1]}" for m in history if isinstance(m, (list, tuple)) and len(m) >= 2]
        history_text = "\n".join(parts) if parts else "None"
    prompt = f"""You are about to search Laravel documentation to answer a user question.
Conversation so far:\n{history_text}\n\nCurrent question: {question}\n\nReply with ONLY a short, specific search query (one line) that would find the right docs. No other text."""
    out = llm.invoke([HumanMessage(content=prompt)])
    return (out.content or question).strip().split("\n")[0].strip()

def rerank_chunks(question, chunks):
    if len(chunks) <= 1:
        return chunks
    prompt = f"Question: {question}\n\nRank these chunks by relevance (most relevant first). Reply with only the chunk numbers in order, comma-separated (e.g. 3,1,2,4).\n\nChunks:\n"
    for i, doc in enumerate(chunks):
        prompt += f"\n--- Chunk {i + 1} ---\n{doc.page_content[:800]}\n"
    prompt += "\nReply with only the comma-separated numbers, e.g. 2,5,1,3,4"
    out = llm.invoke([HumanMessage(content=prompt)])
    numbers = re.findall(r"\d+", (out.content or "").strip())
    order = [int(n) - 1 for n in numbers if 1 <= int(n) <= len(chunks)]
    seen = set()
    ordered = []
    for i in order:
        if i not in seen:
            seen.add(i)
            ordered.append(chunks[i])
    for i, c in enumerate(chunks):
        if i not in seen:
            ordered.append(c)
    return ordered[:FINAL_K] if ordered else chunks[:FINAL_K]

def merge_chunks(a, b):
    seen = set()
    out = []
    for doc in a + b:
        key = doc.page_content
        if key not in seen:
            seen.add(key)
            out.append(doc)
    return out

def fetch_context(question, history=None):
    chunks = fetch_context_unranked(question)
    if USE_REWRITE and history is not None:
        rewritten = rewrite_query(question, history)
        if rewritten and rewritten != question:
            chunks = merge_chunks(chunks, fetch_context_unranked(rewritten))
    if USE_RERANK and chunks:
        chunks = rerank_chunks(question, chunks)
    else:
        chunks = chunks[:FINAL_K]
    return chunks

In [ ]:
def answer_question(question, history=None):
    history = history or []
    docs = fetch_context(question, history)
    context = "\n\n".join(f"From {doc.metadata.get('source', 'doc')}:\n{doc.page_content}" for doc in docs)
    messages = [SystemMessage(content=SYSTEM_PROMPT.format(context=context))]
    for m in history:
        if isinstance(m, (list, tuple)) and len(m) >= 2:
            messages.append(HumanMessage(content=m[0]))
            messages.append(AIMessage(content=m[1]))
    messages.append(HumanMessage(content=question))
    response = llm.invoke(messages)
    return response.content or "", docs

def chat_fn(message, history):
    reply, _ = answer_question(message, history)
    return reply

gr.ChatInterface(
    fn=chat_fn,
    title="Laravel Docs RAG",
    description="Ask questions about Laravel. Uses optional query rewrite and rerank (Week 5 style).",
).launch(inline=True)